## Proyecto MathIA
### Clasificación desbalanceada para el diagnóstico de diferencias cognitivas y de aprendizaje en la enseñanza de las matemáticas
#### Entrenamiento

Este programa realiza los siguientes pasos, simulando una **puesta en producción**:

FASE DE ENTRENAMIENTO:

- Carga un conjunto de datos supervisado de estudiantes en fomato .csv para tareas de clasificación binaria o multiclase.
- Ajusta los hiper-parámetros de un clasificador [EasyEnsembleClassifier](https://imbalanced-learn.org/stable/references/generated/imblearn.ensemble.EasyEnsembleClassifier.html).
- Entrena el modelo de clasificación con el conjunto completo de datos.
- Calcula e imprime en pantalla el [balanced accuracy](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html) del modelo en full training.
- Guarda el modelo, el codificador de etiquetas y la lista con la estructura de columnas en ficheros serializados .pkl para la fase de predicción en producción.

**Detalles en:**

Ana F. Hernández, Andrej Franulic, Javier A. Salazar, José Palma, Fernando Jiménez (2025), Imbalanced Classification for Diagnosing Learning and Cognitive Differences in Mathematics Education, XI Conference - European Congress of Methodology (EAM2025), La Laguna, Santa Cruz de Tenerife, Spain

##### Autor:
Fernando Jiménez Barrionuevo

In [1]:
import numpy as np
import pandas as pd
import warnings
import joblib  # Librería estándar para guardar y cargar los artefactos del modelo
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# =============================================================================
# FASE 1: ENTRENAMIENTO
# =============================================================================

# Leer el dataset desde un archivo CSV
file_name = 'transformed_data_Binary.csv'
df = pd.read_csv(file_name)
print(f"Fichero de datos cargado: {file_name}")

# Eliminar identificador
df = df.drop(columns=["Identificador"])

# Preparar las características (X) y la variable objetivo (y)
X = df.iloc[:, :-1]  # Todas las columnas excepto la última
y_raw = df.iloc[:, -1]   # Solo la última columna

# Aplicar One-Hot Encoding al dataset de entrenamiento
X = pd.get_dummies(X)

# Guardar la lista exacta de columnas que ve el modelo (para la fase predicción)
columnas_entrenamiento = X.columns.tolist()

le = LabelEncoder()
y = le.fit_transform(y_raw)
y = pd.Series(y)

# Verificar el mapeo de clases:
mapeo = {clase: int(codigo) for clase, codigo in zip(le.classes_, le.transform(le.classes_))}
print(f"Mapeo de clases: {mapeo}")

# Detectar clases con solo una muestra y duplicar
class_counts = y.value_counts()
classes_to_duplicate = class_counts[class_counts == 1].index

for c in classes_to_duplicate:
    idx = y[y == c].index[0]
    X = pd.concat([X, X.loc[[idx]]], ignore_index=True)
    y = pd.concat([y, pd.Series([y.loc[idx]])], ignore_index=True)

# Ajustar hiper-parámetros con GridSearchCV (3-fold CV estratificado sin warnings) y entrenar modelo en full training
print("Ajustando hiper-parámetros y entrenando el modelo ...")

param_grid = {
    'n_estimators': [10, 50, 100],
    'sampling_strategy': ['auto', 'majority', 'not minority']
}

cv_estratificado = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
base_clf = EasyEnsembleClassifier(random_state=42)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    grid_search = GridSearchCV(
        estimator=base_clf,
        param_grid=param_grid,
        scoring='balanced_accuracy',
        cv=cv_estratificado,
        n_jobs=-1
    )
    grid_search.fit(X, y)

# Imprimir mejores hiper-parámetros
clf = grid_search.best_estimator_
print(f"Mejores hiper-parámetros: {grid_search.best_params_}")

# Calcular e imprimir balanced accuracy 
train_pred = clf.predict(X)
baac = balanced_accuracy_score(y, train_pred)
print(f"Balanced accuracy en full training: {baac:.4f}")

# =============================================================================
# EXPORTACIÓN DE ARTEFACTOS PARA LA FASE DE PREDICCIÓN EN PRODUCCIÓN
# =============================================================================
# Guardar los 3 elementos críticos en ficheros externos independientes:
# Modelo, codificador de etiquetas y lista con la estructura de columnas
joblib.dump(clf, 'modelo.pkl')
joblib.dump(le, 'label_encoder.pkl')
joblib.dump(columnas_entrenamiento, 'columnas_entrenamiento.pkl')
print("Artefactos guardados con éxito en archivos externos (.pkl).")

print("\nEjecución completada correctamente.")

Fichero de datos cargado: transformed_data_Binary.csv
Mapeo de clases: {'D0': 0, 'D1': 1}
Ajustando hiper-parámetros y entrenando el modelo ...
Mejores hiper-parámetros: {'n_estimators': 100, 'sampling_strategy': 'auto'}
Balanced accuracy en full training: 0.8120
Artefactos guardados con éxito en archivos externos (.pkl).

Ejecución completada correctamente.
